# 01 - Setup & validate Lakehouse

Confirms a schema-enabled Lakehouse is attached, creates the medallion **schemas**
(`bronze`, `silver`, `gold`), and checks the landing Parquet files are present.
Upload `data/parquet/*.parquet` to the Lakehouse **Files/landing/** folder first
(the `10_provision_fabric.ps1` script does this automatically).

## Create medallion schemas

The Lakehouse must be **schema-enabled** (10_provision_fabric.ps1 creates it that way).
`silver` is created but intentionally left empty for now.

In [ ]:
for schema in ('bronze', 'silver', 'gold'):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}')
    print('schema ready:', schema)

In [ ]:
import notebookutils
LANDING = 'Files/landing'
try:
    files = [f.name for f in notebookutils.fs.ls(LANDING)]
except Exception as ex:
    raise RuntimeError(
        f"Could not list {LANDING}. Make sure a default Lakehouse is attached "
        f"(the 20_load_data.ps1 job sets executionData.configuration.defaultLakehouse) "
        f"and that the parquet files were uploaded by 10_provision_fabric.ps1. Root cause: {ex}")
print(f'{len(files)} files in {LANDING}:')
for f in sorted(files):
    print('  ', f)

In [ ]:
# Sanity check: warn (do not fail) if any expected landing file is missing,
# so the pipeline can still proceed for the tables that are present.
expected = ['dim_account', 'dim_customer', 'dim_customer_device', 'dim_device', 'dim_geography', 'dim_plan', 'dim_product', 'dim_promotion', 'fact_appointment', 'fact_contact', 'fact_coverage', 'fact_feedback', 'fact_invoice', 'fact_invoice_line', 'fact_offer', 'fact_outage', 'fact_service_metric', 'fact_subscription', 'fact_usage_data', 'fact_usage_voice', 'fact_work_order', 'ml_churn_score', 'ml_crosssell_reco']
present = {f.replace('.parquet','') for f in files}
missing = [t for t in expected if t not in present]
if missing:
    print('WARNING - missing landing files:', missing)
else:
    print('All expected landing files present.')